# 机器学习概述（上）线性回归与多项式回归

本节按机器学习实践的**五要素**组织：数据、模型、学习准则、优化算法、评估指标。先用线性回归把五要素串成最小 pipeline，再用多项式回归直观看到**模型容量**对欠拟合 / 过拟合的影响，最后用 L2 正则压住过拟合。

> 线性回归本身在 chap3 有更系统的处理；本节侧重五要素框架与正则化效果。

## 机器学习五要素 ↔ PyTorch 对照

| 要素 | 角色 | PyTorch 对应物 |
|---|---|---|
| 数据 | 训练 / 验证 / 测试集 | `torch.utils.data.Dataset`、`DataLoader` |
| 模型 | $f(x;\theta)$ | `nn.Module`（如 `nn.Linear`） |
| 学习准则 | 损失函数 + 正则项 | `nn.MSELoss`、优化器的 `weight_decay` |
| 优化算法 | 更新 $\theta$ | `torch.optim.SGD`、`torch.optim.Adam` 等 |
| 评估指标 | 度量模型效果 | 自定义函数 / `torchmetrics` |

本章在合成数据上演示，因此把 Dataset/DataLoader 推迟到 chap2 下的波士顿房价案例再展开。

## 1. 最小线性回归：用五要素走通一遍

### 数据：合成 $y = 1.2 x + 0.5 + \epsilon$

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)

def linear_data(n, w=1.2, b=0.5, noise=2.0, interval=(-10, 10)):
    x = torch.empty(n, 1).uniform_(*interval)
    y = w * x + b + noise * torch.randn_like(x)
    return x, y

X_train, y_train = linear_data(100)
X_test,  y_test  = linear_data(50)
X_large, y_large = linear_data(5000)

xs = torch.linspace(-10, 10, 200).unsqueeze(1)
plt.scatter(X_train.numpy(), y_train.numpy(), s=15, alpha=0.6, label='train (100)')
plt.scatter(X_test.numpy(),  y_test.numpy(),  s=15, alpha=0.6, label='test (50)')
plt.plot(xs.numpy(), (1.2*xs + 0.5).numpy(), 'k-', lw=1, label='underlying')
plt.legend(); plt.xlabel('x'); plt.ylabel('y'); plt.show()

### 模型 + 准则 + 优化：闭式解 vs SGD

线性回归的最小二乘解有闭式：$\hat w = (X^\top X)^{-1} X^\top y$。

- 数值实现首选 `torch.linalg.lstsq`（内部用 SVD/QR，比直接求逆稳）。
- 也用 `nn.Linear` + `SGD` 对比一下，验证两者收敛到同一组参数。

In [ ]:
def lstsq_fit(X, y):
    """闭式最小二乘。把偏置当作多出来的一列常数 1。"""
    X1 = torch.cat([X, torch.ones_like(X[:, :1])], dim=1)
    sol = torch.linalg.lstsq(X1, y).solution
    return sol[:-1].squeeze(), sol[-1].squeeze()

w_c, b_c = lstsq_fit(X_train, y_train)
w_l, b_l = lstsq_fit(X_large, y_large)
print(f'closed-form  100  : w={w_c.item():.4f}  b={b_c.item():.4f}')
print(f'closed-form  5000 : w={w_l.item():.4f}  b={b_l.item():.4f}')

In [ ]:
torch.manual_seed(0)
model = nn.Linear(1, 1)
opt = optim.SGD(model.parameters(), lr=0.005)
for _ in range(2000):
    loss = nn.functional.mse_loss(model(X_train), y_train)
    opt.zero_grad(); loss.backward(); opt.step()
print(f'SGD          100  : w={model.weight.item():.4f}  b={model.bias.item():.4f}')

### 评估：训练 / 测试 MSE

In [ ]:
def mse(y_hat, y):
    return ((y_hat - y) ** 2).mean().item()

def pred(x, w, b):
    return w * x + b

print(f'train mse (100) : {mse(pred(X_train, w_c, b_c), y_train):.4f}')
print(f'train mse (5000): {mse(pred(X_large, w_l, b_l), y_large):.4f}')
print(f'test  mse (100) : {mse(pred(X_test,  w_c, b_c), y_test):.4f}')
print(f'test  mse (5000): {mse(pred(X_test,  w_l, b_l), y_test):.4f}')

样本量从 100 提到 5000 后，参数更接近真值 (w=1.2, b=0.5)，测试 MSE 也下降——这是"更多数据 → 估计方差更小"的最直接体现。

## 2. 多项式回归：模型容量与欠/过拟合

把 $x$ 经过多项式基函数 $\phi(x) = [x, x^2, \ldots, x^M]^\top$ 映射到高维，再在 $\phi(x)$ 上做线性回归，就是 $M$ 阶多项式拟合。这是"线性方法做非线性"的最经典例子。

目标函数：$y = \sin(2\pi x) + \epsilon$，训练集只有 15 个点。

In [ ]:
torch.manual_seed(0)

def sin_data(n, noise=0.5):
    x = torch.rand(n, 1)
    y = torch.sin(2 * math.pi * x) + noise * torch.randn_like(x)
    return x, y

X_train, y_train = sin_data(15)
X_test,  y_test  = sin_data(10)

x_under = torch.linspace(0, 1, 200).unsqueeze(1)
y_under = torch.sin(2 * math.pi * x_under)

plt.scatter(X_train.numpy(), y_train.numpy(), s=30, label='train')
plt.plot(x_under.numpy(), y_under.numpy(), 'k-', lw=1, label=r'$\sin(2\pi x)$')
plt.legend(); plt.show()

### 多项式基函数 + 不同阶数拟合

In [ ]:
def poly_basis(x, degree):
    """[N, 1] -> [N, degree]：列依次为 x, x^2, …, x^M。
    degree=0 时返回空列；偏置项由 fit_predict 里独立拼上去。"""
    if degree == 0:
        return torch.empty(x.shape[0], 0)
    return torch.cat([x ** k for k in range(1, degree + 1)], dim=1)

def fit_predict(X_train, y_train, X_eval, degree, lam=0.0):
    """多项式基 + (带正则的) 闭式最小二乘拟合，再在 X_eval 上预测。"""
    Phi_t = poly_basis(X_train, degree)
    Phi_e = poly_basis(X_eval,  degree)
    Phi_t1 = torch.cat([Phi_t, torch.ones_like(X_train[:, :1])], dim=1)
    Phi_e1 = torch.cat([Phi_e, torch.ones_like(X_eval[:,  :1])], dim=1)
    D = Phi_t1.shape[1]
    I = torch.eye(D); I[-1, -1] = 0    # 偏置不参与正则化
    A = Phi_t1.T @ Phi_t1 + lam * I
    w = torch.linalg.solve(A, Phi_t1.T @ y_train)
    return Phi_e1 @ w

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, M in zip(axes.flat, [0, 1, 3, 8]):
    y_hat = fit_predict(X_train, y_train, x_under, degree=M)
    ax.scatter(X_train.numpy(), y_train.numpy(), s=20)
    ax.plot(x_under.numpy(), y_under.numpy(), 'k-', lw=1, label=r'$\sin(2\pi x)$')
    ax.plot(x_under.numpy(), y_hat.numpy(),  'r-', lw=1.5, label=f'M={M} fit')
    ax.set_ylim(-2, 2); ax.set_title(f'M={M}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

观察：$M=0/1$ 表示能力不足、欠拟合；$M=3$ 已经能很好地近似 $\sin$；$M=8$ 在仅 15 个点上严重过拟合，曲线在数据稀疏处剧烈波动。

### 训练 / 测试误差随阶数变化

In [ ]:
train_errs, test_errs = [], []
for M in range(9):
    y_train_pred = fit_predict(X_train, y_train, X_train, degree=M)
    y_test_pred  = fit_predict(X_train, y_train, X_test,  degree=M)
    train_errs.append(mse(y_train_pred, y_train))
    test_errs.append(mse(y_test_pred,   y_test))

plt.plot(train_errs, '-o', label='train')
plt.plot(test_errs,  '-s', label='test')
plt.xlabel('polynomial degree M'); plt.ylabel('MSE')
plt.yscale('log'); plt.grid(alpha=0.3); plt.legend(); plt.show()
print('train MSE per M:', [round(e, 4) for e in train_errs])
print('test  MSE per M:', [round(e, 4) for e in test_errs])

### L2 正则化：在 M=8 上压住过拟合

ridge 形式的闭式解：$\hat w = (X^\top X + \lambda I)^{-1} X^\top y$。偏置项通常不正则化。

下面用**真实函数曲线 $\sin(2\pi x)$ 上密集采样**作为评估面（而非小的噪声测试集），这样能直接看到 L2 让拟合曲线**更贴近真分布**的效果。

In [ ]:
M, lam = 8, 0.1
y0  = fit_predict(X_train, y_train, x_under, degree=M, lam=0.0)
yr  = fit_predict(X_train, y_train, x_under, degree=M, lam=lam)

# 在干净的 (x_under, y_under) 上算 MSE，反映"拟合曲线 vs 真分布"的距离
print(f'MSE on true sin surface  no-reg  M=8     :  {mse(y0, y_under):.4f}')
print(f'MSE on true sin surface  L2  λ={lam}     :  {mse(yr, y_under):.4f}')

plt.scatter(X_train.numpy(), y_train.numpy(), s=20, label='train')
plt.plot(x_under.numpy(), y_under.numpy(), 'k-',  lw=1,   label=r'$\sin(2\pi x)$')
plt.plot(x_under.numpy(), y0.numpy(),      'r--', lw=1.2, label='M=8, no reg')
plt.plot(x_under.numpy(), yr.numpy(),      'b-',  lw=1.5, label=f'M=8, L2 (λ={lam})')
plt.ylim(-2, 2); plt.legend(); plt.show()

## 小结

- **闭式解**：`torch.linalg.lstsq` 求无正则的最小二乘；带 L2 正则用 `torch.linalg.solve` 解 $(X^\top X + \lambda I) w = X^\top y$。
- **多项式回归 = 基函数变换 + 线性回归**：模型容量随阶数 $M$ 增长；$M$ 过大 + 数据少 → 过拟合。
- **L2 正则**压低系数模长，曲线更平滑。PyTorch 训练时更常用 `torch.optim.*` 的 `weight_decay` 参数实现等效效果。
- chap2 下用 `Runner` 类把"数据 → 模型 → 训练 → 评估 → 保存"封装成一个可复用骨架，并跑通波士顿房价回归。